In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings
import pandas as pd
from sklearn.datasets import load_wine
warnings.filterwarnings('ignore')
%matplotlib inline

#### Домашнее задание № 1
Задача для прогнозирования предсказания возможного дохода
1. Проверьте данные на пропуски
2. Обучите логистическую регрессию
3. Обучите метод опорных векторов
4. Сравните точность двух моделей
5. Напишите выводы и интерпретируйте

In [ ]:
df = pd.read_csv('adult.csv')
df.head(5)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K


In [ ]:
### YOUR CODE: Домашнее задание № 1 (прогноз дохода >50K)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Проверка пропусков
df_adult = df.replace('?', np.nan) #замена ? на Nan, чтобы пандас норм распознавал все
print('Пропуски по столбцам:')
print(df_adult.isna().sum()[lambda s: s > 0])

# Удаление строк с пропусками
df_adult = df_adult.dropna()

# Целевая переменная: 1 если доход > 50 тысяч, иначе 0.
y = (df_adult['income'] == '>50K').astype(int)
X = df_adult.drop(columns=['income'])

# Разделяем признаки на числовые и категориальные.
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

# Препроцессор: числовые масштабируем, категориальные кодируем one-hot.
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

# Разбиваем на train/test со стратификацией по классу.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Логистическая регрессия
logreg = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000)),
]).fit(X_train, y_train)

# Метод опорных векторов
svm = Pipeline([
    ('prep', preprocessor),
    ('model', SVC()),
]).fit(X_train, y_train)

# Сравнение точности
acc_logreg = accuracy_score(y_test, logreg.predict(X_test))
acc_svm = accuracy_score(y_test, svm.predict(X_test))
print(f'Accuracy LogReg: {acc_logreg:.4f}')
print(f'Accuracy SVM:    {acc_svm:.4f}')


Пропуски по столбцам:
workclass         2799
occupation        2809
native-country     857
dtype: int64
Accuracy LogReg: 0.8441
Accuracy SVM:    0.8478


## Выводы

Логистическая регрессия и SVM показали почти одинаковую точность (0.844 против 0.848) и обе уверенно превзошли baseline ( приблизительно 0.75 при предсказании всегда меньше/равно 50K). Признаки дохода хорошо разделяются линейно, поэтому сложная модель почти не даёт выигрыша — логрегрессия предпочтительнее как более быстрая и интерпретируемая.

#### Домашнее задание № 2

1. Из данных исключите объекты класса 2.
2. Отмасштабируйте признаки, используя класс `StandardScaler` с гиперпараметрами по умолчанию.
3. Обучите логистическую регрессию и оцените важность признаков.
4. Укажите название признака, который оказался наименее значимым.
5. Напишите выводы.

Обратите внимание, целевое значение лежит по ключу `'target'`, матрица объекты-признаки лежит по ключу `'data'`

In [ ]:
data = load_wine()


In [ ]:
### YOUR CODE: Домашнее задание № 2 (датасет Wine)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Целевая переменная по ключу 'target', матрица признаков по ключу 'data'.
X = pd.DataFrame(data['data'], columns=data['feature_names'])
y = pd.Series(data['target'], name='target')

# Исключаем объекты класса 2
# Остаётся бинарная задача: 0 и 1.
mask = y != 2
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)
print('Классы после фильтрации:', np.unique(y))
print('Размер выборки:', X.shape)

# Масштабирование признаков StandardScaler

# Это важно: после стандартизации все признаки в одной шкале (среднее 0, std 1),
# поэтому коэффициенты логрегрессии становятся сравнимыми между собой.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Логистическая регрессия + оценка важности признаков
clf = LogisticRegression()
clf.fit(X_scaled, y)
print('Accuracy на обучении:', clf.score(X_scaled, y))

# Важность признака = модуль его коэффициента (данные стандартизированы,
# поэтому |coef| напрямую отражает силу влияния признака).
importance = pd.Series(clf.coef_[0], index=X.columns)
importance_abs = importance.abs().sort_values(ascending=False)
print('Важность признаков (|коэффициент|), по убыванию:')
print(importance_abs)

#  Наименее значимый признак
least_important = importance_abs.index[-1]
print(f'Наименее значимый признак: {least_important}')

Классы после фильтрации: [0 1]
Размер выборки: (130, 13)
Accuracy на обучении: 1.0
Важность признаков (|коэффициент|), по убыванию:
proline                         1.814575
alcohol                         1.540414
alcalinity_of_ash               1.241012
ash                             0.970386
color_intensity                 0.799957
od280/od315_of_diluted_wines    0.627520
malic_acid                      0.494349
flavanoids                      0.329152
magnesium                       0.237613
proanthocyanins                 0.186724
nonflavanoid_phenols            0.174388
hue                             0.150938
total_phenols                   0.035735
dtype: float64
Наименее значимый признак: total_phenols


##Выводы
После исключения класса 2 классы 0 и 1 разделяются идеально (accuracy = 1.0) — это разные сорта вина с различающимся химическим составом. Сильнее всего на классификацию влияют proline, alcohol и alcalinity_of_ash, а наименее значимым оказался total_phenols (коэффициент ≈ 0, признак практически не помогает различать эти два класса). Масштабирование было обязательным условием для корректного сравнения важности признаков.

### Домашнее задание № 3
В этой части мы будем работать с данными UCI Bank Marketing Dataset. Этот датасет содержит информацию о банковском телефонном маркетинге.

Объектом здесь является телефонный звонок потенциальному клиенту с предложением некоторой услуги (утверждается, что это краткосрочный депозит). В качестве признакового описания используются характеристики клиента (образование, брак и т.д.), более подробная информация представлена в файле bank-additional-names.txt. Целевая переменная - ответ клиента (согласился ли он открыть депозит?)

1. Закодируйте категориальные признаки
2. Выберите метрику классификации, которая вам кажется подходящей, и обучите логистическую регрессию
3. Как вы считаете, что для вашего бизнеса важнее — хороший precision или recall модели? Почему?

In [ ]:
df = pd.read_csv('bank-additional-full.csv', sep=';')
df.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [ ]:
### YOUR CODE: Домашнее задание № 3 (Bank Marketing)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Убираем 'duration': по описанию датасета это утечка таргета
# Для реалистичной модели этот признак исключают.
df_bank = df.drop(columns=['duration'])

# Целевая переменная: 1 если клиент открыл депозит ('yes'), иначе 0.
y = (df_bank['y'] == 'yes').astype(int)
X = df_bank.drop(columns=['y'])
print('Доля положительного класса (yes):', round(y.mean(), 3))  # примерно 0.11, следовательно дисбаланс

# Кодирование категориальных признаков
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),  # one-hot кодирование
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Логистическая регрессия и выбор метрики
# Классы сильно несбалансированы (~11% "yes"), поэтому accuracy неинформативна
# (модель "всегда no" даст ~89%). Смотрим на precision/recall/f1 и ROC-AUC,
# а в самой модели используем class_weight='balanced' против дисбаланса.
clf = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced')),
]).fit(X_train, y_train)

pred = clf.predict(X_test)
proba = clf.predict_proba(X_test)[:, 1]

print('Матрица ошибок:')
print(confusion_matrix(y_test, pred))
print('Отчёт классификации:')
print(classification_report(y_test, pred, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba), 3))

Доля положительного класса (yes): 0.113
Матрица ошибок:
[[9382 1583]
 [ 492  900]]
Отчёт классификации:
              precision    recall  f1-score   support

           0      0.950     0.856     0.900     10965
           1      0.362     0.647     0.465      1392

    accuracy                          0.832     12357
   macro avg      0.656     0.751     0.682     12357
weighted avg      0.884     0.832     0.851     12357

ROC-AUC: 0.803


## Выводы
Из-за сильного дисбаланса классов ( приблизительн 11% «yes») accuracy неинформативна, поэтому опирались на recall, F1 и ROC-AUC (0.803 — приемлемое качество). Модель ловит около 65% реально заинтересованных клиентов ценой невысокого precision (приблизительно 36%), что для задачи обзвона оправдано: дешёвый звонок против дорогой упущенной сделки делает recall приоритетной метрикой. Исключение duration дало честную, применимую на практике модель без утечки таргета.